# Analiza podataka koriscenih u treniranju modela

In [1]:
import pandas as pd

### Sirovi podaci iz OpenFOAM

In [3]:
raw_train_dataset = pd.read_csv("data/raw_data/data_train.csv")
raw_valid_dataset = pd.read_csv("data/raw_data/data_valid.csv")
raw_test_dataset = pd.read_csv("data/raw_data/data_test.csv")

In [7]:
raw_train_dataset.describe()

,time,re,x,y,U_x,U_y,p
count,1.576960e+06,1.576960e+06,1.576960e+06,1.576960e+06,1.576960e+06,1.576960e+06,1.576960e+06
mean,5.000000e+00,5.549854e+02,5.000000e-01,5.000000e-01,6.479670e-05,4.350304e-05,-1.221505e-02
std,3.162279e+00,2.640626e+02,2.886400e-01,2.886400e-01,1.657997e-01,1.239648e-01,3.974701e-02
min,0.000000e+00,1.000000e+02,7.812500e-03,7.812500e-03,-3.279960e-01,-6.345330e-01,-1.318330e+00
25%,2.000000e+00,3.387755e+02,2.539062e-01,2.539062e-01,-5.576783e-02,-1.047610e-02,-2.225412e-02
50%,5.000000e+00,5.775510e+02,5.000000e-01,5.000000e-01,-1.496685e-02,2.496075e-03,-3.444605e-03
75%,8.000000e+00,7.795918e+02,7.460938e-01,7.460938e-01,0.000000e+00,5.664795e-02,5.275023e-05
max,1.000000e+01,1.000000e+03,9.921875e-01,9.921875e-01,9.522080e-01,2.951730e-01,2.034840e+00


In [11]:
raw_valid_dataset.describe()

,time,re,x,y,U_x,U_y,p
count,315392.000000,315392.000000,315392.000000,315392.000000,315392.000000,315392.000000,315392.000000
mean,5.000000,572.303207,0.500000,0.500000,0.000065,0.000044,-0.012220
std,3.162283,264.481885,0.288640,0.288640,0.164769,0.123579,0.038078
min,0.000000,173.469388,0.007812,0.007812,-0.327835,-0.634369,-0.711972
25%,2.000000,302.040816,0.253906,0.253906,-0.054823,-0.010169,-0.021991
50%,5.000000,522.448980,0.500000,0.500000,-0.014695,0.002564,-0.003462
75%,8.000000,871.428571,0.746094,0.746094,0.000000,0.056383,0.000044
max,10.000000,926.530612,0.992188,0.992188,0.945962,0.273310,1.403180


In [15]:
raw_test_dataset.describe()

,time,re,x,y,U_x,U_y,p
count,1.441792e+06,1.441792e+06,1.441792e+06,1.441792e+06,1.441792e+06,1.441792e+06,1.441792e+06
mean,5.000000e+00,5.086735e+02,5.000000e-01,5.000000e-01,1.583748e-05,6.557979e-06,-1.295494e-02
std,3.162279e+00,2.656528e+02,2.886664e-01,2.886664e-01,1.716607e-01,1.276941e-01,4.510042e-02
min,0.000000e+00,1.183673e+02,3.906250e-03,3.906250e-03,-3.468320e-01,-6.666490e-01,-2.344710e+00
25%,2.000000e+00,2.974490e+02,2.519531e-01,2.519531e-01,-5.884415e-02,-1.154218e-02,-2.424005e-02
50%,5.000000e+00,4.948980e+02,5.000000e-01,5.000000e-01,-1.578980e-02,2.126115e-03,-3.672755e-03
75%,8.000000e+00,7.153061e+02,7.480469e-01,7.480469e-01,0.000000e+00,5.832170e-02,4.689610e-05
max,1.000000e+01,8.897959e+02,9.960938e-01,9.960938e-01,9.755410e-01,3.164180e-01,3.247080e+00


Podaci su generisani tako da Rejnoldsovi brojevi budu uzorkovani uniformno iz intervala 100 do 1000. Nakon toga se uzorkovane vrijednosti dijele na trening, validacioni i test skup, pri cemu se vodi racuna o stratifikaciji.

In [23]:
train_re = set(raw_train_dataset["re"].astype(float).round(2).unique())
valid_re = set(raw_valid_dataset["re"].astype(float).round(2).unique())
test_re = set(raw_test_dataset["re"].astype(float).round(2).unique())

fmt = lambda values: [f"{v:.2f}" for v in sorted(values)]

print("Re brojevi po skupu")
print("-" * 50)
print(f"Train ({len(train_re)}): {fmt(train_re)}")
print(f"Valid ({len(valid_re)}): {fmt(valid_re)}")
print(f"Test  ({len(test_re)}): {fmt(test_re)}")

print("\nProvjera disjunktnosti")
print("-" * 50)
print(f"Train | Valid: {fmt(train_re & valid_re)}")
print(f"Train | Test : {fmt(train_re & test_re)}")
print(f"Valid | Test : {fmt(valid_re & test_re)}")

is_disjoint = (
    train_re.isdisjoint(valid_re)
    and train_re.isdisjoint(test_re)
    and valid_re.isdisjoint(test_re)
)

print(f"\nSkupovi su međusobno disjunktni: {is_disjoint}")

Re brojevi po skupu
--------------------------------------------------
Train (35): ['100.00', '136.73', '155.10', '191.84', '210.20', '246.94', '265.31', '283.67', '338.78', '357.14', '375.51', '393.88', '412.24', '448.98', '485.71', '504.08', '540.82', '577.55', '595.92', '614.29', '632.65', '651.02', '687.76', '706.12', '724.49', '761.22', '779.59', '797.96', '816.33', '834.69', '908.16', '944.90', '963.27', '981.63', '1000.00']
Valid (7): ['173.47', '302.04', '467.35', '522.45', '742.86', '871.43', '926.53']
Test  (8): ['118.37', '228.57', '320.41', '430.61', '559.18', '669.39', '853.06', '889.80']

Provjera disjunktnosti
--------------------------------------------------
Train | Valid: []
Train | Test : []
Valid | Test : []

Skupovi su međusobno disjunktni: True


Solver dijeli prostor na ćelije. Za generisanje trening i validacionog skupa korišćena je podjela na 64x64 ćelije, a za test skup na 128x128 ćelija.
Pošto se čuvaju vrijednosti samo u centrima ćelija, na ovaj način nijedna prostorna tačka (x,y) iz test skupa se ne nalazi ni u trening ni u validacionom skupu.

In [26]:
train_x = set(raw_train_dataset["x"].astype(float).round(6))
valid_x = set(raw_valid_dataset["x"].astype(float).round(6))
test_x  = set(raw_test_dataset["x"].astype(float).round(6))

train_y = set(raw_train_dataset["y"].astype(float).round(6))
valid_y = set(raw_valid_dataset["y"].astype(float).round(6))
test_y  = set(raw_test_dataset["y"].astype(float).round(6))

print("X vrijednosti")
print("-" * 40)
print(f"Train | Test : {len(train_x & test_x)} | Disjunktni: {train_x.isdisjoint(test_x)}")
print(f"Valid | Test : {len(valid_x & test_x)} | Disjunktni: {valid_x.isdisjoint(test_x)}")

print("\nY vrijednosti")
print("-" * 40)
print(f"Train | Test : {len(train_y & test_y)} | Disjunktni: {train_y.isdisjoint(test_y)}")
print(f"Valid | Test : {len(valid_y & test_y)} | Disjunktni: {valid_y.isdisjoint(test_y)}")

X vrijednosti
----------------------------------------
Train | Test : 0 | Disjunktni: True
Valid | Test : 0 | Disjunktni: True

Y vrijednosti
----------------------------------------
Train | Test : 0 | Disjunktni: True
Valid | Test : 0 | Disjunktni: True


### Uzorkovanje

Pošto su dobijeni trening i validacioni skup preveliki za treniranje i ne mogu da predstave prednost PINN nad običnim modelom, vrši se uzorkovanje iz generisanih podataka kako bi se dobili manji skupovi. Uzorkovanje uzima u obzir važnost rubnih tačaka.

In [41]:
sampled_train_dataset = pd.read_csv(f"data/sampled_data/frac_5/data_train.csv")
sampled_train_dataset.describe()

,time,re,x,y,U_x,U_y,p
count,11820.000000,11820.000000,11820.000000,11820.000000,11820.000000,11820.000000,11820.000000
mean,4.959898,555.005180,0.503171,0.499243,-0.000074,-0.000398,-0.011910
std,3.174868,265.638134,0.288205,0.289874,0.167241,0.122170,0.038719
min,0.000000,100.000000,0.007812,0.007812,-0.323715,-0.626659,-0.323832
25%,2.000000,338.775510,0.257812,0.242188,-0.056350,-0.009620,-0.022081
50%,5.000000,577.551020,0.507812,0.492188,-0.014715,0.002455,-0.003378
75%,8.000000,779.591837,0.757812,0.757812,0.000000,0.054839,0.000055
max,10.000000,1000.000000,0.992188,0.992188,0.950837,0.264953,0.978912


In [40]:
sampled_valid_dataset = pd.read_csv(f"data/sampled_data/frac_1/data_valid.csv")
sampled_valid_dataset.describe()

,time,re,x,y,U_x,U_y,p
count,473.000000,473.000000,473.000000,473.000000,473.000000,473.000000,473.000000
mean,5.243129,570.755490,0.516566,0.520266,-0.003183,-0.001965,-0.013224
std,3.182660,258.227730,0.285342,0.278395,0.171800,0.130524,0.028415
min,0.000000,173.469388,0.007812,0.007812,-0.322576,-0.594661,-0.093894
25%,3.000000,302.040816,0.273438,0.304688,-0.061517,-0.015170,-0.023709
50%,5.000000,522.448980,0.523438,0.523438,-0.017427,0.001725,-0.005843
75%,8.000000,871.428571,0.773438,0.757812,0.000000,0.070115,0.000000
max,10.000000,926.530612,0.992188,0.992188,0.927724,0.248608,0.230252
